## Setup

In [ ]:
!pip -q install spacy nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import math
from collections import defaultdict, Counter

import nltk
from nltk import word_tokenize, pos_tag
from nltk.chunk import ne_chunk
from nltk.tree import Tree

import spacy
from spacy import displacy

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


## Black-box baselines

Let's first find out how spaCy and NLTK do Named Entity Recognition. For this we will define a couple of consistent functions that will do all the job

In [ ]:
def spacy_ner(text: str):
    """
    Run spaCy NER and return (span_text, label) pairs.
    """
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents], doc

def nltk_ner(text: str):
    """
    Run NLTK POS + NE chunker and return (span_text, label) pairs.
    """
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged, binary=False)

    spans = []
    current = []
    current_label = None

    for node in tree:
        if isinstance(node, Tree):
            label = node.label()
            words = [w for (w, _) in node.leaves()]
            spans.append((" ".join(words), label))
        else:
            # plain token, ignore
            pass
    return spans

Let's now use them with a small corpus

In [ ]:
texts = [
    "Apple hired John Smith in Paris in 2024.",
    "I ate an apple in Paris.",
    "Washington announced new measures today.",
    "Amazon opened a new office in Madrid."
]

for t in texts:
    sp, doc = spacy_ner(t)
    nk = nltk_ner(t)

    print("\nTEXT:", t)
    print("spaCy:", sp)
    print("NLTK :", nk)


TEXT: Apple hired John Smith in Paris in 2024.
spaCy: [('Apple', 'ORG'), ('John Smith', 'PERSON'), ('Paris', 'GPE'), ('2024', 'DATE')]
NLTK : [('Apple', 'PERSON'), ('John Smith', 'PERSON'), ('Paris', 'GPE')]

TEXT: I ate an apple in Paris.
spaCy: [('Paris', 'GPE')]
NLTK : [('Paris', 'GPE')]

TEXT: Washington announced new measures today.
spaCy: [('Washington', 'GPE'), ('today', 'DATE')]
NLTK : [('Washington', 'GPE')]

TEXT: Amazon opened a new office in Madrid.
spaCy: [('Amazon', 'ORG'), ('Madrid', 'GPE')]
NLTK : [('Amazon', 'PERSON'), ('Madrid', 'GPE')]


which may be rendered as follows (just to have a clearer visual inspection)

In [ ]:
doc = nlp("Apple hired John Smith in Paris in 2024.")
displacy.render(doc, style="ent", jupyter=True)

## Define our own tag space

We are going to define a small semantic tag set:

Option A (simple per-token labels, no spans): PER, ORG, LOC, DATE, O

Option B (IOB for spans): B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-DATE, I-DATE, O

Below we implement Option B (IOB) because it keeps boundaries explicit, but the HMM code is identical either way.

In [ ]:
TAGS = ["B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-DATE","I-DATE","O"]
START = "<s>"
END = "</s>"
UNK = "<unk>"

## Tiny annotated NER corpus (IOB)

This toy corpus is intentionally small since it is not assumed to be used under performance conditions. Notice how "Apple" changes role by context:

* "Apple hired..." -> organization-like
* "an apple" -> not a named entity (O)

We also include multi-token entities to make boundaries visible.


In [ ]:
train_sents = [
    [("Apple","B-ORG"), ("hired","O"), ("John","B-PER"), ("Smith","I-PER"), ("in","O"), ("Paris","B-LOC"), ("in","O"), ("2024","B-DATE"), (".","O")],
    [("Apple","B-ORG"), ("announced","O"), ("a","O"), ("statement","O"), (".","O")],
    [("I","O"), ("ate","O"), ("an","O"), ("apple","O"), ("in","O"), ("Paris","B-LOC"), (".","O")],
    [("Microsoft","B-ORG"), ("met","O"), ("John","B-PER"), ("in","O"), ("Madrid","B-LOC"), (".","O")],
    [("John","B-PER"), ("visited","O"), ("New","B-LOC"), ("York","I-LOC"), (".","O")],
    [("Amazon","B-ORG"), ("opened","O"), ("an","O"), ("office","O"), ("in","O"), ("Madrid","B-LOC"), (".","O")],
    [("Paris","B-LOC"), ("hosts","O"), ("events","O"), ("in","O"), ("April","B-DATE"), (".","O")],
    [("Steve","B-PER"), ("Jobs","I-PER"), ("founded","O"), ("Apple","B-ORG"), (".","O")],
]

## HMM Training from counts

As usual we train our HMM based on the frequencies of emissions and transitions

In [ ]:
def train_hmm_from_tagged_sents(tagged_sents, tags, start_token=START, end_token=END, unk_token=UNK):
    """
    Build transition and emission counts from token-level annotated sentences.

    Returns:
        transition_counts: dict[prev_tag] -> Counter(next_tag)
        emission_counts: dict[tag] -> Counter(word)
        tag_counts: Counter(tag)
        vocab: set(words)
    """
    transition_counts = defaultdict(Counter)
    emission_counts = defaultdict(Counter)
    tag_counts = Counter()
    vocab = set()

    for sent in tagged_sents:
        prev = start_token

        # START (first tag)
        if len(sent) > 0:
            transition_counts[start_token][sent[0][1]] += 1

        for word, tag in sent:
            vocab.add(word)
            emission_counts[tag][word] += 1
            tag_counts[tag] += 1

            if prev != start_token:
                transition_counts[prev][tag] += 1
            prev = tag

        # last tag (END)
        transition_counts[prev][end_token] += 1

    # ensure UNK exists in emission table
    for t in tags:
        emission_counts[t][unk_token] += 0

    return transition_counts, emission_counts, tag_counts, vocab

Let's use it with the training corpus we just defined

In [ ]:
transition_counts, emission_counts, tag_counts, vocab = train_hmm_from_tagged_sents(train_sents, TAGS)

print("Tag counts:", tag_counts)
print("Vocab size:", len(vocab))

Tag counts: Counter({'O': 30, 'B-LOC': 6, 'B-ORG': 5, 'B-PER': 4, 'I-PER': 2, 'B-DATE': 2, 'I-LOC': 1})
Vocab size: 30


## Smoothed Probabilities

We can now fine the smoothed log probabilities. Remember that we use the add_k approach in both, emissions and transitions, to avoid zero probabilities that may crash the logs (and bias the results)

In [ ]:
def log_prob_transition(prev_tag, next_tag, tags, transition_counts, k=1.0):
    """
    log P(next_tag | prev_tag) with add-k smoothing.
    """
    next_space = list(tags) + [END]
    num = transition_counts[prev_tag][next_tag] + k
    den = sum(transition_counts[prev_tag][t] for t in next_space) + k * len(next_space)
    return math.log(num / den)

def log_prob_emission(tag, word, vocab, emission_counts, k=1.0):
    """
    log P(word | tag) with add-k smoothing + UNK handling.
    """
    w = word if word in vocab else UNK
    num = emission_counts[tag][w] + k
    den = sum(emission_counts[tag][x] for x in vocab) + emission_counts[tag][UNK] + k * (len(vocab) + 1)
    return math.log(num / den)

def prob_from_log(logp):
    return math.exp(logp)

## Readable tables

Let's now define a couple of functions that can show us the top transitions and emissions (just to see what is going on)

In [ ]:
def show_top_transitions(prev_tag, tags, transition_counts, top_k=8, k=1.0):
    candidates = list(tags) + [END]
    scored = [(nxt, prob_from_log(log_prob_transition(prev_tag, nxt, tags, transition_counts, k=k))) for nxt in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop transitions from {prev_tag}:")
    for nxt, p in scored[:top_k]:
        print(f"{prev_tag:>8} -> {nxt:<8}  {p:.3f}")

def show_top_emissions(tag, vocab, emission_counts, top_k=10, k=1.0):
    candidates = list(vocab) + [UNK]
    scored = [(w, prob_from_log(log_prob_emission(tag, w, vocab, emission_counts, k=k))) for w in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop emissions for {tag} (P(word|{tag})):")
    for w, p in scored[:top_k]:
        w_disp = "<UNK>" if w == UNK else w
        print(f"{tag:>8} -> {w_disp:<12}  {p:.3f}")

Let's use them with specific target tags in the trained corpus

In [ ]:
print("\n=== Selected transition structure (context) ===")
show_top_transitions("O", TAGS, transition_counts)
show_top_transitions("B-PER", TAGS, transition_counts)
show_top_transitions("B-LOC", TAGS, transition_counts)

print("\n=== Selected emissions (semantics) ===")
show_top_emissions("B-ORG", vocab, emission_counts)
show_top_emissions("B-PER", vocab, emission_counts)
show_top_emissions("B-LOC", vocab, emission_counts)


=== Selected transition structure (context) ===

Top transitions from O:
       O -> O         0.325
       O -> </s>      0.225
       O -> B-LOC     0.150
       O -> B-PER     0.075
       O -> B-DATE    0.075
       O -> B-ORG     0.050
       O -> I-PER     0.025
       O -> I-ORG     0.025

Top transitions from B-PER:
   B-PER -> I-PER     0.214
   B-PER -> O         0.214
   B-PER -> B-PER     0.071
   B-PER -> B-ORG     0.071
   B-PER -> I-ORG     0.071
   B-PER -> B-LOC     0.071
   B-PER -> I-LOC     0.071
   B-PER -> B-DATE    0.071

Top transitions from B-LOC:
   B-LOC -> O         0.375
   B-LOC -> I-LOC     0.125
   B-LOC -> B-PER     0.062
   B-LOC -> I-PER     0.062
   B-LOC -> B-ORG     0.062
   B-LOC -> I-ORG     0.062
   B-LOC -> B-LOC     0.062
   B-LOC -> B-DATE    0.062

=== Selected emissions (semantics) ===

Top emissions for B-ORG (P(word|B-ORG)):
   B-ORG -> Apple         0.111
   B-ORG -> Amazon        0.056
   B-ORG -> Microsoft     0.056
   B-ORG -> statem

## Viterbi Decoding

We are going to use the standard Viterbi decoding function (same as in the previous document for POS)

In [ ]:
def viterbi_decode(words, tags, transition_counts, emission_counts, vocab, k_trans=1.0, k_emit=1.0):
    """
    Standard Viterbi for an HMM in log-space.

    Returns:
        best_path: list of tags (same length as words)
        best_log_score: float
    """
    if not words:
        return [], 0.0

    dp = []
    back = []

    # init
    dp0, back0 = {}, {}
    for t in tags:
        dp0[t] = log_prob_transition(START, t, tags, transition_counts, k=k_trans) + \
                 log_prob_emission(t, words[0], vocab, emission_counts, k=k_emit)
        back0[t] = START
    dp.append(dp0)
    back.append(back0)

    # recursion
    for i in range(1, len(words)):
        dpi, backi = {}, {}
        for t in tags:
            best_prev, best_score = None, -1e100
            for prev_t in tags:
                score = dp[i-1][prev_t] + \
                        log_prob_transition(prev_t, t, tags, transition_counts, k=k_trans) + \
                        log_prob_emission(t, words[i], vocab, emission_counts, k=k_emit)
                if score > best_score:
                    best_score = score
                    best_prev = prev_t
            dpi[t] = best_score
            backi[t] = best_prev
        dp.append(dpi)
        back.append(backi)

    # terminate
    best_last, best_last_score = None, -1e100
    for t in tags:
        score = dp[-1][t] + log_prob_transition(t, END, tags, transition_counts, k=k_trans)
        if score > best_last_score:
            best_last_score = score
            best_last = t

    # backtrack
    best_path = [best_last]
    for i in range(len(words)-1, 0, -1):
        best_path.append(back[i][best_path[-1]])
    best_path.reverse()

    return best_path, best_last_score

In [ ]:
def run_sentence(sent: str):
    words = sent.split()
    pred_tags, score = viterbi_decode(words, TAGS, transition_counts, emission_counts, vocab, k_trans=1.0, k_emit=1.0)
    print("\nSentence:", sent)
    print("Prediction:", list(zip(words, pred_tags)))
    print("log-score:", round(score, 3))

unknown entity in our toy vocab

In [ ]:
tests = [
    "Apple hired John Smith in Paris in 2024 .",
    "I ate an apple in Paris .",
    "Steve Jobs founded Apple .",
    "Washington announced measures .",
]

for s in tests:
    run_sentence(s)


Sentence: Apple hired John Smith in Paris in 2024 .
Prediction: [('Apple', 'B-ORG'), ('hired', 'O'), ('John', 'B-PER'), ('Smith', 'I-PER'), ('in', 'O'), ('Paris', 'B-LOC'), ('in', 'O'), ('2024', 'O'), ('.', 'O')]
log-score: -37.498

Sentence: I ate an apple in Paris .
Prediction: [('I', 'B-ORG'), ('ate', 'O'), ('an', 'O'), ('apple', 'O'), ('in', 'O'), ('Paris', 'B-LOC'), ('.', 'O')]
log-score: -29.673

Sentence: Steve Jobs founded Apple .
Prediction: [('Steve', 'B-PER'), ('Jobs', 'I-PER'), ('founded', 'O'), ('Apple', 'B-ORG'), ('.', 'O')]
log-score: -23.316

Sentence: Washington announced measures .
Prediction: [('Washington', 'B-ORG'), ('announced', 'O'), ('measures', 'O'), ('.', 'O')]
log-score: -18.963


## Viterbi DP table

As in POS tagging, we can view the log-probabilities from the dynamic programming table use the same function

In [ ]:
def viterbi_debug_table(words, allowed_tags, all_tags,
                       transition_counts, emission_counts, vocab,
                       k_trans=1.0, k_emit=1.0):
    """
    Prints a compact Viterbi DP table:
    rows = allowed_tags, columns = word positions, values = best log-score so far.
    Uses all_tags for probability normalisation (smoothing denominators).
    """

    if not words:
        print("Empty sentence.")
        return

    dp = []
    back = []

    # init
    dp0, back0 = {}, {}
    for t in allowed_tags:
        dp0[t] = (
            log_prob_transition(START, t, all_tags, transition_counts, k=k_trans)
            + log_prob_emission(t, words[0], vocab, emission_counts, k=k_emit)
        )
        back0[t] = START

    dp.append(dp0)
    back.append(back0)

    # recursion
    for i in range(1, len(words)):
        dpi, backi = {}, {}
        for t in allowed_tags:
            best_prev, best_score = None, -1e100
            for prev_t in allowed_tags:
                score = (
                    dp[i-1][prev_t]
                    + log_prob_transition(prev_t, t, all_tags, transition_counts, k=k_trans)
                    + log_prob_emission(t, words[i], vocab, emission_counts, k=k_emit)
                )
                if score > best_score:
                    best_score = score
                    best_prev = prev_t
            dpi[t] = best_score
            backi[t] = best_prev
        dp.append(dpi)
        back.append(backi)

    # terminate
    best_last, best_last_score = None, -1e100
    for t in allowed_tags:
        score = dp[-1][t] + log_prob_transition(t, END, all_tags, transition_counts, k=k_trans)
        if score > best_last_score:
            best_last_score = score
            best_last = t

    # backtrack
    best_path = [best_last]
    for i in range(len(words) - 1, 0, -1):
        best_path.append(back[i][best_path[-1]])
    best_path.reverse()

    # print table
    header = "tag\\pos".ljust(10) + "".join(w.center(12) for w in words)
    print("\nSentence:", " ".join(words))
    print(header)
    print("-" * len(header))

    for t in allowed_tags:
        row = t.ljust(10)
        for i in range(len(words)):
            row += f"{dp[i][t]:12.2f}"
        print(row)

    # local vs global
    local_best = []
    for i in range(len(words)):
        best_t = max(allowed_tags, key=lambda t: dp[i][t])
        local_best.append(best_t)

    print("\nLocal best per step:", list(zip(words, local_best)))
    print("Final Viterbi path :", list(zip(words, best_path)))
    print("Final log-score    :", round(best_last_score, 3))

Used as follows

In [ ]:
allowed = ["B-ORG", "B-PER", "B-LOC", "O"]
sent_demo = "Apple hired John in Paris .".split()

viterbi_debug_table(
    sent_demo,
    allowed_tags=allowed,
    all_tags=TAGS,
    transition_counts=transition_counts,
    emission_counts=emission_counts,
    vocab=vocab,
    k_trans=1.0,
    k_emit=1.0
)


Sentence: Apple hired John in Paris .
tag\pos      Apple       hired        John         in        Paris         .      
----------------------------------------------------------------------------------
B-ORG            -3.48       -9.77      -14.39      -18.79      -22.86      -26.75
B-PER            -5.35       -9.74      -12.57      -18.77      -22.42      -26.73
B-LOC            -5.81       -9.80      -13.32      -18.56      -20.40      -26.78
O                -6.31       -7.81      -13.05      -16.28      -21.51      -23.29

Local best per step: [('Apple', 'B-ORG'), ('hired', 'O'), ('John', 'B-PER'), ('in', 'O'), ('Paris', 'B-LOC'), ('.', 'O')]
Final Viterbi path : [('Apple', 'B-ORG'), ('hired', 'O'), ('John', 'B-PER'), ('in', 'O'), ('Paris', 'B-LOC'), ('.', 'O')]
Final log-score    : -24.785


Note, again, that the point with the log-score is that it is meaningful in the case we use the same sentence in different hyperparameter space (changing the add_k for transmissions and emissions)

## Evaluations

Let's proceed to define a set of helper functions that allow us to find the evaluation metrics

In [ ]:
def iob_to_spans(words, tags):
    """
    Convert IOB tags to spans.
    Returns a set of (start_idx, end_idx_exclusive, ent_type) tuples.
    Assumes tags are like B-PER, I-PER, O.
    """
    spans = set()
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag == "O":
            i += 1
            continue

        if "-" not in tag:
            # if you ever use non-IOB tags, treat as O here
            i += 1
            continue

        prefix, ent_type = tag.split("-", 1)

        if prefix == "B":
            start = i
            i += 1
            # consume I-<type>
            while i < len(tags) and tags[i] == f"I-{ent_type}":
                i += 1
            end = i
            spans.add((start, end, ent_type))
        else:
            # An I- tag without a preceding B- is ill-formed; skip it
            i += 1

    return spans

def span_prf1(true_spans, pred_spans):
    """
    Strict span evaluation.
    """
    tp = len(true_spans & pred_spans)
    fp = len(pred_spans - true_spans)
    fn = len(true_spans - pred_spans)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return precision, recall, f1, tp, fp, fn

def token_accuracy(true_tags, pred_tags):
    correct = sum(t == p for t, p in zip(true_tags, pred_tags))
    return correct / len(true_tags) if true_tags else 0.0

def evaluate_dataset(tagged_sents, tags_universe, transition_counts, emission_counts, vocab,
                     k_trans=1.0, k_emit=1.0, allowed_tags=None):
    """
    Evaluate on a dataset of tagged sentences.
    Prints token accuracy and strict span P/R/F1.
    """
    total_tok_correct = 0
    total_tok = 0

    all_true_spans = set()
    all_pred_spans = set()

    for sent in tagged_sents:
        words = [w for w, _ in sent]
        true_tags = [t for _, t in sent]

        # If you want to restrict decoding to a subset of tags, pass allowed_tags.
        decode_tags = allowed_tags if allowed_tags is not None else tags_universe

        pred_tags, _ = viterbi_decode(
            words,
            decode_tags,
            transition_counts,
            emission_counts,
            vocab,
            k_trans=k_trans,
            k_emit=k_emit
        )

        # Token accuracy
        total_tok_correct += sum(t == p for t, p in zip(true_tags, pred_tags))
        total_tok += len(true_tags)

        # Span metrics (strict)
        true_spans = iob_to_spans(words, true_tags)
        pred_spans = iob_to_spans(words, pred_tags)

        # To get corpus-level P/R/F1, pool spans across sentences with an offset
        # Here we offset by sentence index * large constant to keep them unique
        offset = 10000 * len(all_true_spans)  # simple unique offset trick
        true_spans_off = {(s+offset, e+offset, t) for (s, e, t) in true_spans}
        pred_spans_off = {(s+offset, e+offset, t) for (s, e, t) in pred_spans}

        all_true_spans |= true_spans_off
        all_pred_spans |= pred_spans_off

    acc = total_tok_correct / total_tok if total_tok else 0.0
    p, r, f1, tp, fp, fn = span_prf1(all_true_spans, all_pred_spans)

    print("Token accuracy:", round(acc, 3))
    print("Strict span precision:", round(p, 3))
    print("Strict span recall   :", round(r, 3))
    print("Strict span F1       :", round(f1, 3))
    print("TP / FP / FN         :", tp, fp, fn)

Which can be used to evaluate on the toy training set (just to demonstrate the metrics)

In [ ]:
evaluate_dataset(
    train_sents,
    tags_universe=TAGS,
    transition_counts=transition_counts,
    emission_counts=emission_counts,
    vocab=vocab,
    k_trans=1.0,
    k_emit=1.0
)

Token accuracy: 0.92
Strict span precision: 0.875
Strict span recall   : 0.824
Strict span F1       : 0.848
TP / FP / FN         : 14 2 3


The interpretation of these number is the usual one:

* Token accuracy 0.92 means 92% of individual tokens got the correct IOB label. This can still hide important NER failures because a single boundary error changes only one or two token labels.

* Strict span precision 0.875 means that when the model predicts an entity span, it is correct most of the time (87.5% of the times). Concretely, 2 of the predicted entity spans were wrong, since FP = 2.

* Strict span recall 0.824 means the model missed some true entities (82.4% of the tru entities were properly recognized as entities). Concretely, it failed to recover 3 gold spans, since FN = 3.

* F1 0.848 balances these two and is the main “NER performance” number (just the usual harmonic mean of precision and recall).

Let's see that a boundary mistake can keep token accuracy high while destroying span correctness:

If the gold is “New York City” (B-LOC I-LOC I-LOC) but the model predicts “New York” (B-LOC I-LOC O), only one token is wrong (“City”), yet the entire entity is counted as incorrect under strict evaluation. That is why span F1 drops faster than token accuracy.

***NOTE**: the notation "gold span" is standard NLP shorthand for the ground-truth, manually annotated entity spans, and the name comes from the gold-standard*

Let's see the false positives and negatives from our model

In [ ]:
def evaluate_and_show_errors(tagged_sents, tags_universe, transition_counts, emission_counts, vocab,
                             k_trans=1.0, k_emit=1.0, allowed_tags=None):
    for idx, sent in enumerate(tagged_sents):
        words = [w for w, _ in sent]
        true_tags = [t for _, t in sent]

        decode_tags = allowed_tags if allowed_tags is not None else tags_universe
        pred_tags, _ = viterbi_decode(words, decode_tags, transition_counts, emission_counts, vocab,
                                      k_trans=k_trans, k_emit=k_emit)

        true_spans = iob_to_spans(words, true_tags)
        pred_spans = iob_to_spans(words, pred_tags)

        fp = pred_spans - true_spans
        fn = true_spans - pred_spans

        if fp or fn:
            print("\nSentence:", " ".join(words))
            print("Tokens:", list(zip(words, true_tags, pred_tags)))
            if fp:
                print("False positives (predicted but wrong):", fp)
            if fn:
                print("False negatives (missed true spans):", fn)

In [ ]:
evaluate_and_show_errors(
    train_sents,
    tags_universe=TAGS,
    transition_counts=transition_counts,
    emission_counts=emission_counts,
    vocab=vocab
)


Sentence: Apple hired John Smith in Paris in 2024 .
Tokens: [('Apple', 'B-ORG', 'B-ORG'), ('hired', 'O', 'O'), ('John', 'B-PER', 'B-PER'), ('Smith', 'I-PER', 'I-PER'), ('in', 'O', 'O'), ('Paris', 'B-LOC', 'B-LOC'), ('in', 'O', 'O'), ('2024', 'B-DATE', 'O'), ('.', 'O', 'O')]
False negatives (missed true spans): {(7, 8, 'DATE')}

Sentence: I ate an apple in Paris .
Tokens: [('I', 'O', 'B-ORG'), ('ate', 'O', 'O'), ('an', 'O', 'O'), ('apple', 'O', 'O'), ('in', 'O', 'O'), ('Paris', 'B-LOC', 'B-LOC'), ('.', 'O', 'O')]
False positives (predicted but wrong): {(0, 1, 'ORG')}

Sentence: John visited New York .
Tokens: [('John', 'B-PER', 'B-PER'), ('visited', 'O', 'O'), ('New', 'B-LOC', 'B-LOC'), ('York', 'I-LOC', 'O'), ('.', 'O', 'O')]
False positives (predicted but wrong): {(2, 3, 'LOC')}
False negatives (missed true spans): {(2, 4, 'LOC')}

Sentence: Paris hosts events in April .
Tokens: [('Paris', 'B-LOC', 'B-LOC'), ('hosts', 'O', 'O'), ('events', 'O', 'O'), ('in', 'O', 'O'), ('April', 'B-DA

To understand the outputs consider ('New', 'B-LOC', 'B-LOC'), which means that

• The word is “New”
• The true (reference) tag is B-LOC
• The model predicted B-LOC

Also, the set elements like {(0, 1, 'ORG')} are predicted entity spans of the form (start_index, end_index_exclusive, entity_type), so (0, 1, 'ORG') means:

• Start at token index 0
• End at index 1 (exclusive)
• Entity type = ORG

With this in mind we can interpret the outputs as:

* The model fails to emit B-DATE for tokens that clearly belong to DATE because emission probabilities for DATE are weak (few dates in training, strong O emissions overall, transitions favouring O to O)
* The model predicts B-ORG for “I”, which says that P(B-ORG|START) is too high beacuse many of the training sentences begin with organizations
* The model stopped after "New", which is a structural weakness of the first-order HMM: it cannot strongly enforce span continuation unless it has enough evidence in both transitions and emissions.